# Huber-DML pilot（代码版，尚未运行）

只稳健化第一阶段 outcome nuisance；A=原HGBR平方损失，B=GBR平方损失，C=GBR Huber(alpha=0.90)。主要比较C−B。原DGP、treatment预测、fold、第二阶段及SE保持不变。

在VS Code中选择 `D:/Anoconda/envs/environment2/python.exe` 内核，然后按顺序运行。当前 `STAGE="pilot"`（每场景200次）；改为 `"smoke"` 则只检查每场景2次，并先检查其中前2次。拟合仅在最后的“执行实验”单元启动。

此notebook已包含方法与汇总代码。仍需保留桌面的 `Huber_DML_pilot` 文件夹（原配置及hash清单）和既有DML结果（用于核验A）。所有新输出进入该文件夹的outputs新run目录，不覆盖旧结果。重复运行执行单元前，须先重跑“环境检查与新建run”单元。


### 2026-11-02 更新：R=200，分位数自适应Huber阈值

C使用sklearn原生的分位数自适应Huber：第k折第t轮，delta[k,t]取训练集绝对残差abs(Y−F[t−1](X))的90%分位数。delta随折和轮次变化；alpha=0.90是固定分位水平，不是固定delta，也不是自动搜索alpha。只使用外层train，不读取held-out残差作阈值校准。

原版本已采用同一机制，本次不声称换成了另一种估计量。正式pilot每场景200组，共600组数据、1800条A/B/C记录；先验收前2组，再运行余下198组。boosting仍为100轮，seed中的100/1000倍数不改。代码已更新，未运行；旧输出保存在修改前备份，旧实验目录不变。


In [ ]:
# 用户参数：不提供alpha、DGP或R网格搜索
STAGE = "pilot"     # "smoke": 每场景2次；"pilot": 每场景200次
PILOT_REPLICATIONS = 200
WORKERS = 4

import os
for key in ("OMP_NUM_THREADS", "MKL_NUM_THREADS", "OPENBLAS_NUM_THREADS", "NUMEXPR_NUM_THREADS"):
    os.environ[key] = "1"
import pathlib, json, hashlib, platform, uuid
from datetime import datetime
from types import SimpleNamespace
import numpy as np
import pandas as pd
import scipy, sklearn, joblib
from joblib import Parallel, delayed
from sklearn.ensemble import HistGradientBoostingRegressor, GradientBoostingRegressor
ROOT = pathlib.Path(r"C:/Users/Lenovo/Desktop/Huber_DML_pilot")
NOTEBOOK = ROOT.parent / "Huber_DML_pilot.ipynb"
if STAGE not in ("smoke", "pilot") or not 1 <= WORKERS <= 16:
    raise ValueError("检查STAGE和WORKERS")
args = SimpleNamespace(stage=STAGE, workers=WORKERS)
def sha(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


## 原DGP、A第一阶段与第二阶段SE
以下13个原函数原样提取；仅定义函数，不训练模型。

In [ ]:
# Frozen original definitions; no notebook launch cells or Dukes correction.
import numpy as np
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
P=5
THETA_0=1.0
NORMAL_BASELINE="N(0,1)"
BOOSTING_PARAMS=dict(learning_rate=0.05,max_iter=100,max_leaf_nodes=15,min_samples_leaf=20,l2_regularization=1.0)

def ar1_covariance(p: int, rho: float = 0.5) -> np.ndarray:
    """构造 Sigma[j,k] = rho**|j-k| 的 AR(1) 型协方差矩阵。"""
    index = np.arange(p)
    return rho ** np.abs(index[:, None] - index[None, :])

def nuisance_functions(X: np.ndarray, dgp_name: str):
    """返回指定 DGP 的 m0(X) 与 g0(X)。"""
    x1, x2, x3, x4, x5 = X.T

    if dgp_name == "DGP1_linear":
        m0 = 0.80 * x1 - 0.50 * x2 + 0.30 * x3 + 0.20 * x4 - 0.10 * x5
        g0 = 1.00 * x1 + 0.50 * x2 - 0.50 * x3 + 0.25 * x4 + 0.10 * x5
    elif dgp_name == "DGP2_nonlinear":
        # x2**2 - 1 做中心化，因为标准正态变量满足 E[X^2]=1
        m0 = (
            0.60 * np.sin(x1)
            + 0.35 * (x2**2 - 1.0)
            + 0.25 * x3 * x4
            - 0.20 * np.tanh(x5)
        )
        g0 = (
            1.20 * np.sin(x1)
            + 0.70 * (x2**2 - 1.0)
            + 0.50 * x3 * x4
            + 0.30 * np.tanh(x5)
        )
    else:
        raise ValueError(f"未知 DGP: {dgp_name}")
    return m0, g0

def standardized_student_t(rng: np.random.Generator, df: float, size: int) -> np.ndarray:
    """生成均值 0、方差 1 的 Student-t 误差；要求 df > 2。"""
    if df <= 2:
        raise ValueError("Student-t 自由度必须大于 2，才能标准化为单位方差。")
    return rng.standard_t(df, size=size) * np.sqrt((df - 2.0) / df)

def generate_data_student(n: int, error_case, dgp_name: str, seed: int, theta0: float = THETA_0):
    """生成一个 Student-t 或 N(0,1) 结果误差下的部分线性模型样本。"""
    rng = np.random.default_rng(seed)
    sigma_x = ar1_covariance(P, rho=0.5)
    X = rng.multivariate_normal(np.zeros(P), sigma_x, size=n)
    m0, g0 = nuisance_functions(X, dgp_name)

    V = rng.normal(0.0, 1.0, size=n)
    if error_case == NORMAL_BASELINE:
        U = rng.normal(0.0, 1.0, size=n)
    else:
        U = standardized_student_t(rng, df=float(error_case), size=n)
    D = m0 + V
    Y = theta0 * D + g0 + U
    return X, D, Y, U

def make_nuisance_model(kind: str, seed: int):
    """根据估计量类型创建条件均值或条件中位数学习器。"""
    common = dict(BOOSTING_PARAMS)
    common["random_state"] = seed
    if kind == "mean":
        return HistGradientBoostingRegressor(loss="squared_error", **common)
    if kind == "median":
        return HistGradientBoostingRegressor(loss="quantile", quantile=0.5, **common)
    raise ValueError("kind 必须是 'mean' 或 'median'。")

def fit_mean_dml(d_res: np.ndarray, y_res: np.ndarray):
    """第二阶段残差 OLS，并由正交得分的经验影响函数计算标准误。"""
    jacobian = np.mean(d_res**2)
    if jacobian <= 1e-12:
        return np.nan, np.nan
    theta_hat = np.mean(d_res * y_res) / jacobian
    score = d_res * (y_res - theta_hat * d_res)
    influence = score / jacobian
    se_hat = np.sqrt(np.var(influence, ddof=1) / len(d_res))
    return float(theta_hat), float(se_hat)

def lognormal_ar1_covariance(p: int, rho: float = 0.5) -> np.ndarray:
    index = np.arange(p)
    return rho ** np.abs(index[:, None] - index[None, :])

def lognormal_nuisance_functions(X: np.ndarray, dgp_name: str):
    x1, x2, x3, x4, x5 = X.T
    if dgp_name == "DGP1_linear":
        m0 = 0.80*x1 - 0.50*x2 + 0.30*x3 + 0.20*x4 - 0.10*x5
        g0 = 1.00*x1 + 0.50*x2 - 0.50*x3 + 0.25*x4 + 0.10*x5
    elif dgp_name == "DGP2_nonlinear":
        m0 = (0.60*np.sin(x1) + 0.35*(x2**2 - 1.0)
              + 0.25*x3*x4 - 0.20*np.tanh(x5))
        g0 = (1.20*np.sin(x1) + 0.70*(x2**2 - 1.0)
              + 0.50*x3*x4 + 0.30*np.tanh(x5))
    else:
        raise ValueError(f"未知 DGP: {dgp_name}")
    return m0, g0

def standardized_centered_lognormal(rng: np.random.Generator, sigma: float, size: int):
    """生成理论均值 0、方差 1 的右偏 lognormal 误差。"""
    if sigma <= 0:
        raise ValueError("lognormal sigma 必须大于 0。")
    z = rng.normal(0.0, 1.0, size=size)
    w = np.exp(sigma * z)
    mean_w = np.exp(0.5 * sigma**2)
    sd_w = np.sqrt(np.expm1(sigma**2) * np.exp(sigma**2))
    return (w - mean_w) / sd_w

def generate_data_lognormal(n: int, error_case, dgp_name: str, seed: int, theta0: float = THETA_0):
    rng = np.random.default_rng(seed)
    sigma_x = lognormal_ar1_covariance(P, rho=0.5)
    X = rng.multivariate_normal(np.zeros(P), sigma_x, size=n)
    m0, g0 = lognormal_nuisance_functions(X, dgp_name)
    V = rng.normal(0.0, 1.0, size=n)
    if error_case == NORMAL_BASELINE:
        U = rng.normal(0.0, 1.0, size=n)
    else:
        U = standardized_centered_lognormal(rng, float(error_case), n)
    D = m0 + V
    Y = theta0 * D + g0 + U
    return X, D, Y, U

def generate_data(n, error_case, dgp_name, seed, theta0=THETA_0, error_family="student_t"):
    if error_family == "lognormal":
        return generate_data_lognormal(n, error_case, dgp_name, seed, theta0)
    if error_family in ("student_t", "gaussian"):
        return generate_data_student(n, "N(0,1)" if error_family == "gaussian" else error_case,
                                     dgp_name, seed, theta0)
    raise ValueError(f"Unknown error family: {error_family}")

def make_data(config, scenario, replication):
    seed = config["base_seed"] + 100000 * config["legacy_dgp_id"] + 1000 * scenario["legacy_error_id"] + replication
    family = "lognormal" if scenario["error_family"] == "lognormal" else "student_t"
    data = generate_data(config["n"], scenario["error_case"], config["dgp"], seed, theta0=config["theta0"], error_family=family)
    folds = list(KFold(config["n_folds"], shuffle=True, random_state=seed + config["fold_seed_offset"]).split(data[0]))
    return data, folds, seed

def initial_a(X, D, Y, folds, seed):
    d_hat, y_hat = np.empty_like(D), np.empty_like(Y)
    for k, (train, test) in enumerate(folds):
        model_d = make_nuisance_model("mean", seed + 100 * k + 1)
        model_y = make_nuisance_model("mean", seed + 100 * k + 2)
        model_d.fit(X[train], D[train])
        model_y.fit(X[train], Y[train])
        d_hat[test] = model_d.predict(X[test])
        y_hat[test] = model_y.predict(X[test])
    return d_hat, y_hat


## 三组方法及逐次诊断
B/C同训练集、同随机状态，仅loss不同。所有方法使用同一份m_hat；A与旧保存结果逐次核验。

In [ ]:
"""Outcome-only Huber comparison; original DGP, initial A and IF-SE are imported."""
import hashlib,json,pathlib,time,warnings
import numpy as np
from sklearn.ensemble import GradientBoostingRegressor
from threadpoolctl import threadpool_limits

# Fixed quantile level; sklearn updates delta on TRAIN residuals at EVERY boosting stage.
HUBER_QUANTILE = 0.90
GBR_COMMON=dict(learning_rate=.05,n_estimators=100,max_leaf_nodes=15,min_samples_leaf=20,
                max_depth=None,subsample=1.0)
METHODS=['A','B','C']

def json_safe(value):
    if isinstance(value,dict):return {k:json_safe(v) for k,v in value.items()}
    if isinstance(value,list):return [json_safe(v) for v in value]
    if isinstance(value,np.generic):return json_safe(value.item())
    if isinstance(value,float) and not np.isfinite(value):return None
    return value

def finite(x):
    if not np.isfinite(x).all():raise FloatingPointError('Nonfinite array or statistic')

def outcome_gbr(X,Y,folds,cfseed,loss):
    prediction=np.full(len(Y),np.nan);seen=np.zeros(len(Y),dtype=int)
    for k,(train,test) in enumerate(folds):
        if np.intersect1d(train,test).size:raise ValueError('Fold overlap')
        model=GradientBoostingRegressor(loss=loss,alpha=HUBER_QUANTILE,random_state=cfseed+100*k+2,**GBR_COMMON)
        model.fit(X[train],Y[train])
        prediction[test]=model.predict(X[test]);seen[test]+=1
    if not np.all(seen==1):raise ValueError('OOF observations not covered exactly once')
    finite(prediction)
    return prediction

def run_replication(config,scenario,rep_id,output,legacy_run):
    output=pathlib.Path(output);started=time.perf_counter();rows=[];captured=[]
    base=dict(scenario=scenario['name'],dgp=config['dgp'],p=config['p'],n=config['n'],rep_id=rep_id)
    stem=f"{scenario['name']}_{rep_id:04d}";arrays={}
    with threadpool_limits(limits=1),warnings.catch_warnings(record=True) as captured:
        warnings.simplefilter('always')
        try:
            (X,D,Y,epsilon),folds,seed=make_data(config,scenario,rep_id)
            cfseed=seed+config['fold_seed_offset'];fold_ids=np.full(len(D),-1,dtype=np.int64)
            seen=np.zeros(len(D),dtype=int)
            for k,(train,test) in enumerate(folds):
                assert len(np.intersect1d(train,test))==0
                assert len(train)+len(test)==len(D)
                fold_ids[test]=k;seen[test]+=1
            assert np.all(seen==1)
            base.update(seed=seed,crossfit_seed=cfseed,fold_hash=hashlib.sha256(fold_ids.tobytes()).hexdigest(),
                        epsilon_variance=float(np.var(epsilon,ddof=1)),epsilon_max_abs=float(abs(epsilon).max()))
            finite(np.array([base['epsilon_variance'],base['epsilon_max_abs']]))
            m_hat,ell_A=initial_a(X,D,Y,folds,cfseed) # treatment fitted once, shared below
            finite(np.r_[m_hat,ell_A,D,Y])
            oracle=lognormal_nuisance_functions if scenario['error_family']=='lognormal' else nuisance_functions
            m0,g0=oracle(X,config['dgp']);ell0=config['theta0']*m0+g0
            base['treatment_rmse']=float(np.sqrt(np.mean((m_hat-m0)**2)))
            base['treatment_prediction_hash']=hashlib.sha256(m_hat.tobytes()).hexdigest()
            arrays.update(X=X,D=D,Y=Y,epsilon=epsilon,fold_ids=fold_ids,m_hat=m_hat,m0=m0,ell0=ell0)
        except Exception as error:
            rows=[dict(base,method=m,status='failed',error=f'{type(error).__name__}: {error}') for m in METHODS]
        if not rows:
            for method in METHODS:
                params=(config['hgbr_params_seed_zero'].copy() if method=='A' else
                        GradientBoostingRegressor(loss='squared_error' if method=='B' else 'huber',alpha=HUBER_QUANTILE,**GBR_COMMON).get_params())
                params['random_state']='crossfit_seed + 100*fold + 2'
                row=dict(base,method=method,learner_params=json.dumps(params,sort_keys=True))
                try:
                    ell=ell_A if method=='A' else outcome_gbr(X,Y,folds,cfseed,'squared_error' if method=='B' else 'huber')
                    theta,se=fit_mean_dml(D-m_hat,Y-ell) # exact legacy second stage
                    finite(np.array([theta,se]));assert se>0,'Nonpositive SE'
                    low,high=theta-1.96*se,theta+1.96*se
                    error=ell-ell0;quantiles=np.quantile(error,[.01,.05,.25,.5,.75,.95,.99])
                    row.update(status='ok',theta_hat=theta,se=se,CI_low=low,CI_high=high,
                               covered=bool(low<=config['theta0']<=high),ci_length=high-low,
                               t_stat=(theta-config['theta0'])/se,
                               outcome_rmse=float(np.sqrt(np.mean(error**2))),outcome_mae=float(np.mean(abs(error))),
                               outcome_error_mean=float(error.mean()),outcome_error_median=float(np.median(error)))
                    row.update({f'outcome_error_q{q}':float(v) for q,v in zip(['01','05','25','50','75','95','99'],quantiles)})
                    finite(np.array([row['t_stat'],row['outcome_rmse'],row['outcome_mae'],low,high,*quantiles]))
                    arrays['ell_'+method]=ell
                    if method=='A':
                        task_path=pathlib.Path(legacy_run)/'tasks'/f'{stem}.json'
                        task=json.loads(task_path.read_text(encoding='utf-8'))
                        old=next(r for r in task['rows'] if r['method']=='original_mean_dml')
                        array_path=pathlib.Path(legacy_run)/task['arrays_file']
                        assert hashlib.sha256(array_path.read_bytes()).hexdigest()==task['arrays_sha256']
                        with np.load(array_path,allow_pickle=False) as saved:
                            md=float(np.max(abs(m_hat-saved['g_hat'])));ld=float(np.max(abs(ell-saved['ell_hat'])))
                            same_folds=np.array_equal(fold_ids,saved['fold_ids'])
                        row.update(legacy_m_maxdiff=md,legacy_ell_maxdiff=ld,
                                   legacy_theta_difference=theta-old['theta_hat'],legacy_se_difference=se-old['theta_se'])
                        row['legacy_match']=bool(same_folds and md<1e-12 and ld<1e-12 and
                                                 abs(theta-old['theta_hat'])<1e-12 and abs(se-old['theta_se'])<1e-12)
                        if not row['legacy_match']:raise ValueError('A differs from saved baseline: inspect legacy_* fields')
                except Exception as error:
                    row.update(status='failed',error=f'{type(error).__name__}: {error}',covered=None)
                rows.append(row)
    warning_text=[str(w.message) for w in captured]
    for row in rows:row.update(warnings=json.dumps(warning_text,ensure_ascii=False),elapsed_seconds=time.perf_counter()-started)
    np.savez_compressed(output/'arrays'/f'{stem}.npz',**arrays)
    rows=json_safe(rows)
    result=dict(rows=rows,seed=base.get('seed'),scenario=scenario['name'],rep_id=rep_id)
    path=output/'tasks'/f'{stem}.json'
    path.write_text(json.dumps(result,ensure_ascii=False,indent=2,allow_nan=False),encoding='utf-8')
    return rows


## 汇总、配对比较与报告函数
只定义函数。Huber不保证估计条件均值；偏态误差尤其需要检查位置偏移。跨场景gain仅作描述，不强加独立性假设。

In [ ]:
"""Summaries are generated only from a user-launched run; no synthetic results."""
import json,pathlib
import numpy as np
import pandas as pd

def summarize(output):
    output=pathlib.Path(output);cfg=json.loads((output/'config.json').read_text(encoding='utf-8'))
    df=pd.read_csv(output/'replications.csv')
    for c in ['theta_hat','se','covered','ci_length','t_stat','outcome_rmse','outcome_mae','outcome_error_mean','treatment_rmse']:
        if c not in df:df[c]=np.nan
    records=[];pairs=[];pair_summaries=[]
    for scenario in [s['name'] for s in cfg['scenarios']]:
        for method in ['A','B','C']:
            group=df[(df.scenario==scenario)&(df.method==method)]
            valid=group[(group.status=='ok')&np.isfinite(group.t_stat)]
            n=len(valid);theta=valid.theta_hat;T=valid.t_stat
            coverage=float(valid.covered.astype(float).mean()) if n else np.nan
            records.append(dict(scenario=scenario,method=method,planned=cfg['replications'],attempted=len(group),valid=n,
                                failed=int((group.status!='ok').sum()),not_attempted=cfg['replications']-len(group),
                                nonfinite_T=int((~np.isfinite(group.t_stat)).sum()),
                                bias=float(theta.mean()-cfg['theta0']),empirical_sd=float(theta.std(ddof=1)),
                                mean_se=float(valid.se.mean()),rmse=float(np.sqrt(np.mean((theta-cfg['theta0'])**2))),
                                coverage=coverage,coverage_mcse=np.sqrt(coverage*(1-coverage)/n) if n else np.nan,
                                mean_ci_length=float(valid.ci_length.mean()),mean_T=float(T.mean()),sd_T=float(T.std(ddof=1)),
                                p_left=float((T < -1.96).mean()) if n else np.nan,p_right=float((T>1.96).mean()) if n else np.nan,
                                outcome_rmse=float(valid.outcome_rmse.mean()),outcome_mae=float(valid.outcome_mae.mean()),
                                outcome_error_mean=float(valid.outcome_error_mean.mean()),treatment_rmse=float(valid.treatment_rmse.mean())))
        bb=df[(df.scenario==scenario)&(df.method=='B')].set_index('rep_id')
        cc=df[(df.scenario==scenario)&(df.method=='C')].set_index('rep_id')
        local=[]
        for rep in sorted(set(bb.index)&set(cc.index)):
            b,c=bb.loc[rep],cc.loc[rep];ok=b.status==c.status=='ok'
            row=dict(scenario=scenario,rep_id=rep,valid_pair=ok)
            if ok:
                assert b.fold_hash==c.fold_hash and b.treatment_prediction_hash==c.treatment_prediction_hash
                row.update(covered_C_minus_B=int(c.covered)-int(b.covered),
                           theta_C_minus_B=c.theta_hat-b.theta_hat,T_C_minus_B=c.t_stat-b.t_stat,
                           outcome_rmse_C_minus_B=c.outcome_rmse-b.outcome_rmse,
                           outcome_mae_C_minus_B=c.outcome_mae-b.outcome_mae,
                           outcome_error_mean_C_minus_B=c.outcome_error_mean-b.outcome_error_mean)
                local.append(row)
            pairs.append(row)
        for metric in ['covered_C_minus_B','theta_C_minus_B','T_C_minus_B','outcome_rmse_C_minus_B','outcome_mae_C_minus_B','outcome_error_mean_C_minus_B']:
            values=np.array([row[metric] for row in local],float)
            pair_summaries.append(dict(scenario=scenario,metric=metric,valid_pairs=len(values),
                                       mean_difference=float(values.mean()) if len(values) else np.nan,
                                       paired_mcse=float(values.std(ddof=1)/np.sqrt(len(values))) if len(values)>1 else np.nan))
    summary=pd.DataFrame(records);paired=pd.DataFrame(pair_summaries)
    summary.to_csv(output/'summary.csv',index=False)
    pd.DataFrame(pairs).to_csv(output/'paired_replications.csv',index=False)
    paired.to_csv(output/'paired_comparisons.csv',index=False)
    df[df.status!='ok'].to_csv(output/'failures.csv',index=False)
    df[df.warnings!='[]'].to_csv(output/'warnings.csv',index=False)
    indexed=summary.set_index(['scenario','method']);tails=[]
    for scenario in ['t2.1','lognormal2']:
        losses={m:indexed.loc[('gaussian',m),'coverage']-indexed.loc[(scenario,m),'coverage'] for m in ['A','B','C']}
        tails.append(dict(scenario=scenario,loss_tail_A=losses['A'],loss_tail_B=losses['B'],loss_tail_C=losses['C'],
                          gain_robust=losses['B']-losses['C'],absolute_coverage_C=indexed.loc[(scenario,'C'),'coverage'],
                          inference='Descriptive only: cross-scenario coupling not assumed independent or paired for MCSE'))
    pd.DataFrame(tails).to_csv(output/'tail_deterioration.csv',index=False)
    plot(summary,output,cfg)
    table='|场景|方法|有效/计划|Coverage|MCSE|Bias|Mean(T)|SD(T)|Outcome RMSE|\n|---|---|---|---|---|---|---|---|---|\n'
    for row in records:
        table+=f"|{row['scenario']}|{row['method']}|{row['valid']}/{row['planned']}|{row['coverage']:.3f}|{row['coverage_mcse']:.3f}|{row['bias']:.4f}|{row['mean_T']:.3f}|{row['sd_T']:.3f}|{row['outcome_rmse']:.3f}|\n"
    paired_text='\n'.join(f"- {r.scenario}: {r.metric} = {r.mean_difference:.5f}，配对MCSE={r.paired_mcse:.5f}，有效配对{r.valid_pairs}。" for r in paired.itertuples())
    tail_text='\n'.join(f"- {r['scenario']}: loss_tail(B)={r['loss_tail_B']:.3f}，loss_tail(C)={r['loss_tail_C']:.3f}，gain_robust={r['gain_robust']:.3f}，C绝对coverage={r['absolute_coverage_C']:.3f}。" for r in tails)
    report=f'''# Huber outcome DML {cfg['stage']} 报告

本次计划每场景{cfg['replications']}次，已尝试{len(df)//3}个数据集。失败方法记录{int((df.status!='ok').sum())}条；未尝试数和失败数明确保留，统计表仅对有效结果计算，不把失败当作未覆盖，也不更换seed。

{table}

重点比较C−B（相同GBR架构，仅loss不同）；C−A同时包含架构变化，不能全部归因于Huber。

{paired_text}

{tail_text}

coverage和gain单位均为比例；0.01为1个百分点。跨场景gain仅描述，不报告未验证耦合下的显著性。R=200最坏coverage MCSE约3.54个百分点，不能因小幅变化宣布有效；smoke结果不能作为效果证据。结果用于方向筛选，不自动扩展R、DGP或alpha。

Huber总体目标未必是条件均值，尤其偏态lognormal可能引入位置偏移。预测误差均以ell0=theta0*m0+g0评估，未作oracle校准；真值仅用于数据生成与评估，不进入模型训练或第二阶段估计。当前独立加性误差下，常数位置偏移在正确treatment nuisance等条件下未必影响总体识别，但有限样本偏差与SE仍需检查，不能声称保留全部DML理论保证。

仅改第一阶段并不保证第二阶段稳健。如果nuisance RMSE改善而coverage未改善，需要结合score/SE诊断，不能直接概括为Huber无效。本轮未加入第二阶段Huber、Dukes或均值校准。

![Coverage](coverage_comparison.png)zh

![Outcome nuisance RMSE](nuisance_rmse.png)

逐次指标见replications.csv，配对指示差见paired_replications.csv；fold、全部OOF预测和原始数据在arrays/；配置与源码hash在config.json。A逐次旧结果复现检查在legacy_*列。
'''
    (output/'pilot_report.md').write_text(report,encoding='utf-8')

def plot(summary,output,cfg):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    scenarios=['gaussian','t2.1','lognormal2']
    for metric,name,title in [('coverage','coverage_comparison.png','95% Wald coverage'),('outcome_rmse','nuisance_rmse.png','Outcome OOF RMSE against true conditional mean')]:
        fig,ax=plt.subplots(figsize=(9,4.5),layout='constrained')
        for j,method in enumerate(['A','B','C']):
            vals=summary[summary.method==method].set_index('scenario').loc[scenarios,metric].to_numpy()
            ax.bar(np.arange(3)+(j-1)*.24,vals,.24,label={'A':'A: HGBR squared','B':'B: GBR squared','C':'C: GBR Huber (adaptive delta, q=0.90)'}[method])
        if metric=='coverage':ax.axhline(.95,ls='--',color='black',label='Nominal 95%');ax.set_ylim(0,1.05)
        ax.set(xticks=np.arange(3),xticklabels=scenarios,ylabel='Proportion' if metric=='coverage' else 'Response units',
               title=f'{title}; n=1000, planned R={cfg["replications"]} per scenario')
        ax.legend(fontsize=8);fig.savefig(output/name,dpi=160);plt.close(fig)


## 环境检查与新建run
本单元检查版本、原源码hash和参数，创建新的输出目录，但不拟合模型。修改STAGE后，请从参数单元重新运行。

In [ ]:
config=json.loads((ROOT/'legacy_config.json').read_text(encoding='utf-8'))
manifest=json.loads((ROOT/'source_manifest.json').read_text(encoding='utf-8'))
# Refuse silent environment drift. No dependency upgrade is performed.
for name,actual in [('numpy',np.__version__),('scipy',scipy.__version__),('sklearn',sklearn.__version__),('joblib',joblib.__version__)]:
    if actual!=config['environment'][name]:raise RuntimeError(f'{name} version differs: {actual}; expected {config["environment"][name]}')
import ast
for path,digest in manifest['files_sha256'].items():
    source_path=pathlib.Path(path)
    if source_path.suffix.lower()=='.ipynb':
        original_nb=json.loads(source_path.read_text(encoding='utf-8'))
        original_code='\n'.join(''.join(c['source']) for c in original_nb['cells'] if c['cell_type']=='code')
        code_hash=hashlib.sha256(ast.dump(ast.parse(original_code),include_attributes=False).encode()).hexdigest()
        if code_hash!=manifest['notebook_code_ast_sha256']:
            raise RuntimeError('Original notebook computation code changed: '+path)
    elif sha(source_path)!=digest:
        raise RuntimeError('Source changed since packaging: '+path)
expected=config['hgbr_params_seed_zero']
actual=HistGradientBoostingRegressor(loss='squared_error',random_state=0,learning_rate=.05,max_iter=100,
            max_leaf_nodes=15,min_samples_leaf=20,l2_regularization=1.0).get_params()
if actual!=expected:raise RuntimeError('Original HGBR default parameters have changed')
b=GradientBoostingRegressor(loss='squared_error',alpha=HUBER_QUANTILE,**GBR_COMMON).get_params()
c=GradientBoostingRegressor(loss='huber',alpha=HUBER_QUANTILE,**GBR_COMMON).get_params()
assert {k for k in b if b[k]!=c[k]}=={'loss'}
output=ROOT/'outputs'/(datetime.now().strftime('%Y%m%d_%H%M%S')+'_'+args.stage+'_'+uuid.uuid4().hex[:8])
output.mkdir(parents=True,exist_ok=False)
for folder in ['tasks','arrays']:(output/folder).mkdir()
config.update(huber_threshold_rule='per_stage_training_absolute_residual_quantile',
              huber_quantile=HUBER_QUANTILE,
              huber_delta_is_adaptive=True,
              notebook_sha256=sha(NOTEBOOK),stage=args.stage,replications=2 if args.stage=='smoke' else PILOT_REPLICATIONS,workers=args.workers,
              gbr_B=b,gbr_C=c,run_id=output.name,python=platform.python_version(),
              source_hashes={p.name:sha(p) for p in ROOT.glob('*.py')},
              seed_rule='base_seed + 100000*legacy_dgp_id + 1000*legacy_error_id + rep_id',
              note='New A/B/C definitions; no Dukes correction. Smoke repetitions reused within pilot.')
(output/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')

_experiment_started = False
print("Output:", output)


## 执行实验
**运行本单元才会开始拟合。** smoke失败时保留记录并停止；pilot先验证前2次，再运行余下198次。失败不会换seed重试。

In [ ]:
if _experiment_started:
    raise RuntimeError("该run已启动过；请先重跑环境检查与新建run单元，避免覆盖")
_experiment_started = True
rows=[]
def batch(reps):
    jobs=[(s,r) for s in config['scenarios'] for r in reps]
    result=Parallel(n_jobs=args.workers,backend='loky')(
        delayed(run_replication)(config,s,r,output,manifest['legacy_run']) for s,r in jobs)
    for group in result:rows.extend(group)
    pd.DataFrame(rows).to_csv(output/'replications.csv',index=False)
batch(range(2))
# Strict smoke gate. Failures are retained; never replace seeds or omit them silently.
if any(r['status']!='ok' for r in rows):
    summarize(output)
    raise RuntimeError(f'Smoke failed; pilot not launched. See {output}')
if args.stage=='pilot':batch(range(2,config['replications']))
summarize(output)
print('Saved:',output)

## 查看结果
报告只有运行实验后才会生成；本次交付没有任何模拟结果。

In [ ]:
from IPython.display import display, Markdown, Image
display(pd.read_csv(output / "summary.csv"))
display(pd.read_csv(output / "paired_comparisons.csv"))
display(pd.read_csv(output / "tail_deterioration.csv"))
display(Markdown((output / "pilot_report.md").read_text(encoding="utf-8")))
display(Image(filename=str(output / "coverage_comparison.png")))
display(Image(filename=str(output / "nuisance_rmse.png")))
